# Jaccpot GPU Runtime Benchmark (Radix Tree)

This notebook benchmarks Jaccpot runtime on GPU using yggdrax radix-tree settings tuned for throughput.

Included plots:
- `Runtime scaling with particle count (full vs prepared split)`
- `prepare_state component breakdown`

## Large-N Runtime Toggle
These benchmarks now support the dedicated `large_n` runtime path. Set `GPU_DEVICE = "9"` and `RUNTIME_PATH = "large_n"` below for the new path, and uses the canonical large_n runtime path.

In [ ]:
import os
GPU_DEVICE = "9"
RUNTIME_PATH = "large_n"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_DEVICE
print(f"CUDA_VISIBLE_DEVICES={GPU_DEVICE}, RUNTIME_PATH={RUNTIME_PATH}")

def inject_runtime_path(kwargs):
    updated = dict(kwargs)
    updated["runtime_path"] = RUNTIME_PATH
    return updated


In [ ]:
import os

# --- Option A: automatic GPU selection with autocvd ---
USE_AUTOCVD = False
AUTOCVD_NUM_GPUS = 1
AUTOCVD_LEAST_USED = True
AUTOCVD_EXCLUDE = []

# --- Option B: manual selection (set to string like '0' or '0,1') ---
MANUAL_CUDA_VISIBLE_DEVICES = "1"  # None #"1,2,3,4,5,6,7,8,9"#None

if MANUAL_CUDA_VISIBLE_DEVICES is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = MANUAL_CUDA_VISIBLE_DEVICES
    print("Set CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])
elif USE_AUTOCVD:
    try:
        from autocvd import autocvd

        autocvd(
            num_gpus=AUTOCVD_NUM_GPUS,
            least_used=AUTOCVD_LEAST_USED,
            exclude=AUTOCVD_EXCLUDE,
        )
        print(
            "autocvd selected CUDA_VISIBLE_DEVICES =",
            os.environ.get("CUDA_VISIBLE_DEVICES"),
        )
    except ImportError:
        print(
            "autocvd is not installed. Install it or set MANUAL_CUDA_VISIBLE_DEVICES."
        )
else:
    print(
        "Using existing CUDA visibility:",
        os.environ.get("CUDA_VISIBLE_DEVICES", "<all visible>"),
    )

# Index precision switch (must be set before importing JAX/jaccpot/yggdrax).
INDEX_PRECISION = "int32"  # choose from: "int32", "int64"
os.environ.setdefault("JACCPOT_INDEX_PRECISION", INDEX_PRECISION)
os.environ.setdefault("YGGDRAX_INDEX_PRECISION", INDEX_PRECISION)
print("Index precision:", os.environ.get("JACCPOT_INDEX_PRECISION"))

# GPU memory-fragmentation / graph-memory safeguards.
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
if "--xla_gpu_enable_command_buffer=" not in os.environ.get("XLA_FLAGS", ""):
    existing_xla_flags = os.environ.get("XLA_FLAGS", "").strip()
    command_buffer_off = "--xla_gpu_enable_command_buffer="
    os.environ["XLA_FLAGS"] = (
        f"{existing_xla_flags} {command_buffer_off}".strip()
        if existing_xla_flags
        else command_buffer_off
    )

# Runtime cache limits (reduce transient cache memory pressure for capacity runs).
os.environ.setdefault("JACCPOT_OPERATOR_CACHE_MAX", "256")
os.environ.setdefault("JACCPOT_GROUPED_OPERATOR_CACHE_MAX", "16")
os.environ.setdefault("JACCPOT_GROUPED_SEGMENT_CACHE_MAX", "16")
os.environ.setdefault("JACCPOT_GROUPED_OPERATOR_CACHE_ENTRY_MAX_BYTES", str(16 * 1024 * 1024))
os.environ.setdefault("JACCPOT_GROUPED_OPERATOR_CACHE_TOTAL_MAX_BYTES", str(64 * 1024 * 1024))
os.environ.setdefault("JACCPOT_GROUPED_SEGMENT_CACHE_ENTRY_MAX_BYTES", str(8 * 1024 * 1024))
os.environ.setdefault("JACCPOT_GROUPED_SEGMENT_CACHE_TOTAL_MAX_BYTES", str(32 * 1024 * 1024))
print(
    "Runtime cache limits:",
    {
        "op_max": os.environ.get("JACCPOT_OPERATOR_CACHE_MAX"),
        "grp_op_max": os.environ.get("JACCPOT_GROUPED_OPERATOR_CACHE_MAX"),
        "grp_seg_max": os.environ.get("JACCPOT_GROUPED_SEGMENT_CACHE_MAX"),
        "grp_op_total_mb": int(os.environ.get("JACCPOT_GROUPED_OPERATOR_CACHE_TOTAL_MAX_BYTES", "0")) // (1024 * 1024),
        "grp_seg_total_mb": int(os.environ.get("JACCPOT_GROUPED_SEGMENT_CACHE_TOTAL_MAX_BYTES", "0")) // (1024 * 1024),
    },
)



In [ ]:
import json
from dataclasses import replace
import pathlib
import subprocess
import sys
import inspect
import time

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = pathlib.Path.cwd().resolve()
if not (REPO_ROOT / "jaccpot").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) in sys.path:
    sys.path.remove(str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT))

from jaccpot import (
    FMMAdvancedConfig,
    FMMPreset,
    FarFieldConfig,
    FastMultipoleMethod,
    NearFieldConfig,
    RuntimePolicyConfig,
    TreeConfig,
)
from yggdrax import Tree, compute_tree_geometry
from yggdrax.interactions import DualTreeTraversalConfig, build_interactions_and_neighbors

from examples import benchmark_utils as bench_utils

In [ ]:
# GPU sanity check
all_devices = jax.devices()
gpu_devices = [d for d in all_devices if d.platform == "gpu"]
print("JAX backend:", jax.default_backend())
print("Visible devices:", all_devices)
if not gpu_devices:
    raise RuntimeError(
        "No GPU device visible to JAX. Ensure CUDA/ROCm is configured before running this notebook."
    )
print("Using GPU:", gpu_devices[0])

## Benchmark Configuration

In [ ]:
# Particle-count sweep for scaling plots (includes your target N=131072)
runtime_particle_counts = [8192, 16384, 32768, 65536, 131072, 262144, 524288]#, 1048576, 2097152, 4194304, 8388608, 16777216]
runtime_leaf_size = 64
runtime_max_order = 4
runtime_runs = 3
runtime_warmup = 1
runtime_isolate_process_per_n = True
runtime_autotune_cache_path = REPO_ROOT / "benchmarks" / "m2l_autotune_cache.json"
runtime_traversal_autotune_cache_path = REPO_ROOT / "benchmarks" / "traversal_autotune_cache.json"
runtime_enable_traversal_autotune = True
runtime_traversal_autotune_probe_n = 131072
runtime_traversal_candidates = [
    dict(max_pair_queue=131072, process_block=128, max_interactions_per_node=1024, max_neighbors_per_leaf=256),
    dict(max_pair_queue=131072, process_block=256, max_interactions_per_node=2048, max_neighbors_per_leaf=512),
    dict(max_pair_queue=262144, process_block=256, max_interactions_per_node=4096, max_neighbors_per_leaf=1024),
    dict(max_pair_queue=524288, process_block=512, max_interactions_per_node=4096, max_neighbors_per_leaf=1024),
]


runtime_benchmark_scope = "steady_eval"
runtime_worker_autotune_traversal = True
runtime_worker_autotune_nearfield_chunk = True
runtime_nearfield_chunk_candidates = [64, 128, 256, 512]
runtime_runtime_autotune_cache_path = REPO_ROOT / "benchmarks" / "runtime_worker_autotune_cache.json"


# Memory-first large-N overrides (kept strict to maximize fit at high N).
runtime_memory_first_enabled = True
runtime_memory_first_m2l_chunk_size = 1024
runtime_memory_first_nearfield_edge_chunk_size = 128
runtime_memory_analysis_enabled = True
runtime_memory_analysis_particles = 25_000 #100_000
runtime_memory_analysis_setups = ["fiducial"]
runtime_memory_analysis_include_full = False

runtime_memory_first_advanced = FMMAdvancedConfig(
    tree=TreeConfig(
        tree_type="radix",
        mode="lbvh",
        leaf_target=64,
        refine_local=False,
        max_refine_levels=0,
        aspect_threshold=16.0,
    ),
    farfield=FarFieldConfig(
        grouped_interactions=True,
        mode="class_major",
        rotation="solidfmm",
        m2l_chunk_size=int(runtime_memory_first_m2l_chunk_size),
        l2l_chunk_size=None,
        streamed_far_pairs=True,
        mixed_order=False,
        mixed_order_min_order=None,
    ),
    nearfield=NearFieldConfig(
        mode="bucketed",
        edge_chunk_size=int(runtime_memory_first_nearfield_edge_chunk_size),
        precompute_scatter_schedules=False,
    ),
    runtime=RuntimePolicyConfig(
        host_refine_mode="off",
        jit_tree=True,
        jit_traversal=True,
        memory_objective="minimum_memory",
        traversal_config=None,
        pair_process_block=None,
        enable_interaction_cache=False,
        retain_traversal_result=False,
        retain_interactions=False,
        autotune_m2l_chunk=True,
    ),
    mac_type="dehnen",
    dehnen_radius_scale=1.0,
)

# GPU-oriented precision for large-N profiling
runtime_precision = "fp32"
runtime_working_dtype = jnp.float32
runtime_adaptive_p_gears = (2, 3, 4)
runtime_adaptive_error_model = "dehnen_degree"
runtime_adaptive_eps = 5e-4
runtime_adaptive_mac_force_scale_mode = "prev"
autotune_warmup_counts = [65536, 131072]
apples_particle_counts = [65536, 131072, 262144, 524288]
apples_runs = 3
apples_warmup = 1
accuracy_particle_counts = [4096, 16384, 65536]
accuracy_direct_target_cap = 2048
accuracy_direct_target_chunk = 256
accuracy_direct_source_chunk = 4096
runtime_key = jax.random.PRNGKey(0)
softening = 1e-3

# Compare a fiducial large-N preset against adaptive-order + error-controlled MAC.
runtime_fmm_kwargs_fiducial = dict(
    preset=FMMPreset.LARGE_N_GPU,
    basis="solidfmm",
    precision=runtime_precision,
    theta=0.6,
    softening=softening,
    working_dtype=runtime_working_dtype,
    adaptive_order=False,
)
if runtime_memory_first_enabled:
    runtime_fmm_kwargs_fiducial["advanced"] = runtime_memory_first_advanced

runtime_fmm_kwargs_adaptive = runtime_fmm_kwargs_fiducial.copy()
runtime_fmm_kwargs_adaptive.update(
    adaptive_order=True,
    p_gears=runtime_adaptive_p_gears,
    adaptive_error_model=runtime_adaptive_error_model,
    adaptive_eps=runtime_adaptive_eps,
    mac_force_scale_mode=runtime_adaptive_mac_force_scale_mode,
)
runtime_setups = {
    "fiducial": runtime_fmm_kwargs_fiducial,
    "adaptive_error_mac": runtime_fmm_kwargs_adaptive,
}
# Prepared-state runtime memory inventory (byte accounting from object tree).
runtime_prepared_memory_particles = [25_000, 50_000, 100_000]
runtime_prepared_memory_setups = ["fiducial"]

# End-to-end peak GPU memory scaling (warm prepare/evaluate on tuned setups).
runtime_peak_memory_enabled = True
runtime_peak_memory_particle_counts = runtime_particle_counts
runtime_peak_memory_setups = ["fiducial"]
runtime_peak_memory_poll_interval_s = 0.02
runtime_peak_memory_csv_path = REPO_ROOT / "benchmarks" / "runtime_peak_memory_scaling.csv"
runtime_peak_memory_trace_csv_path = REPO_ROOT / "benchmarks" / "runtime_peak_memory_trace.csv"

# Memory-pressure sweep (find stable non-OOM runtime knobs).
runtime_memory_pressure_enabled = True
runtime_memory_pressure_setup = "fiducial"
runtime_memory_pressure_particles = [100_000, 131_072, 200_000, 262_144, 350_000, 500_000]
runtime_memory_pressure_m2l_chunks = [2048, 1024, 512, 256]
runtime_memory_pressure_nearfield_chunks = [256, 128, 64]
runtime_memory_pressure_runs = 1
runtime_memory_pressure_warmup = 0


In [ ]:
for setup_name, setup_kwargs in runtime_setups.items():
    setup_probe = FastMultipoleMethod(**setup_kwargs)
    print(f"{setup_name} resolved memory path:", bench_utils.resolved_large_n_memory_path_report(setup_probe))
    _release_runtime_memory(setup_probe)


## Runtime Collection Helpers

In [ ]:
def _evaluate_prepared_kwargs(fmm):
    params = inspect.signature(fmm.evaluate_prepared_state).parameters
    if "jit_traversal" in params:
        return {"jit_traversal": True}
    return {}


def _release_runtime_memory(fmm):
    clear_fn = getattr(fmm, "clear_runtime_caches", None)
    if callable(clear_fn):
        clear_fn(clear_jax_compilation=True)
        return
    if hasattr(fmm, "clear_prepared_state_cache"):
        fmm.clear_prepared_state_cache()
    jax.clear_caches()


def _iter_notebook_fmm_instances(obj, *, seen=None):
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return
    seen.add(obj_id)
    if isinstance(obj, FastMultipoleMethod):
        yield obj
        return
    if isinstance(obj, dict):
        for value in obj.values():
            yield from _iter_notebook_fmm_instances(value, seen=seen)
        return
    if isinstance(obj, (list, tuple, set)):
        for value in obj:
            yield from _iter_notebook_fmm_instances(value, seen=seen)


def _release_notebook_runtime_state():
    import gc

    released = set()
    for name, value in list(globals().items()):
        if name.startswith("__"):
            continue
        for fmm in _iter_notebook_fmm_instances(value):
            fmm_id = id(fmm)
            if fmm_id in released:
                continue
            try:
                _release_runtime_memory(fmm)
            except Exception:
                pass
            released.add(fmm_id)
    gc.collect()
    jax.clear_caches()


def _serialize_fmm_kwargs_for_worker(fmm_kwargs):
    probe_fmm = FastMultipoleMethod(**fmm_kwargs)
    advanced = probe_fmm.advanced
    traversal_cfg = advanced.runtime.traversal_config
    traversal_payload = None
    if traversal_cfg is not None:
        traversal_payload = {
            "process_block": int(traversal_cfg.process_block),
            "max_neighbors_per_leaf": int(traversal_cfg.max_neighbors_per_leaf),
            "max_interactions_per_node": int(traversal_cfg.max_interactions_per_node),
            "max_pair_queue": int(traversal_cfg.max_pair_queue),
        }
    preset_value = fmm_kwargs.get("preset", "fast")
    if hasattr(preset_value, "value"):
        preset_value = preset_value.value
    payload = {
        "preset": str(preset_value),
        "basis": str(fmm_kwargs.get("basis", "solidfmm")),
        "theta": float(fmm_kwargs.get("theta", 0.6)),
        "softening": float(fmm_kwargs.get("softening", 1e-3)),
        "working_dtype": str(jnp.dtype(getattr(probe_fmm._impl, "working_dtype", jnp.float32))),
        "tree_type": str(advanced.tree.tree_type),
        "leaf_target": int(advanced.tree.leaf_target),
        "farfield_rotation": str(advanced.farfield.rotation),
        "farfield_mode": str(advanced.farfield.mode),
        "grouped_interactions": bool(advanced.farfield.grouped_interactions),
        "streamed_far_pairs": advanced.farfield.streamed_far_pairs,
        "mixed_order": bool(advanced.farfield.mixed_order),
        "mixed_order_min_order": advanced.farfield.mixed_order_min_order,
        "nearfield_mode": str(advanced.nearfield.mode),
        "nearfield_edge_chunk_size": int(advanced.nearfield.edge_chunk_size),
        "precompute_scatter_schedules": bool(advanced.nearfield.precompute_scatter_schedules),
        "pair_process_block": (
            None
            if advanced.runtime.pair_process_block is None
            else int(advanced.runtime.pair_process_block)
        ),
        "jit_traversal": bool(advanced.runtime.jit_traversal),
        "traversal_config": traversal_payload,
        "enable_interaction_cache": bool(advanced.runtime.enable_interaction_cache),
        "retain_traversal_result": bool(advanced.runtime.retain_traversal_result),
        "retain_interactions": bool(advanced.runtime.retain_interactions),
        "autotune_m2l_chunk": bool(advanced.runtime.autotune_m2l_chunk),
        "adaptive_order": bool(getattr(probe_fmm._impl, "adaptive_order", False)),
        "p_gears": [int(v) for v in getattr(probe_fmm._impl, "p_gears", tuple())],
        "adaptive_error_model": str(getattr(probe_fmm._impl, "adaptive_error_model", "tail_proxy")),
        "adaptive_eps": (
            None
            if getattr(probe_fmm._impl, "adaptive_eps", None) is None
            else float(getattr(probe_fmm._impl, "adaptive_eps"))
        ),
        "mac_force_scale_mode": str(getattr(probe_fmm._impl, "mac_force_scale_mode", "prev")),
        "mac_type": str(advanced.mac_type),
    }
    _release_runtime_memory(probe_fmm)
    return payload


def _build_worker_command(mode, *, num_particles, leaf_size, max_order, runs, warmup, dtype, fmm_kwargs, emit_ready_marker=False):
    worker_script = REPO_ROOT / "examples" / "benchmark_gpu_radix_worker.py"
    payload = _serialize_fmm_kwargs_for_worker(fmm_kwargs)
    cmd = [
        sys.executable,
        str(worker_script),
        "--mode",
        str(mode),
        "--num-particles",
        str(int(num_particles)),
        "--leaf-size",
        str(int(leaf_size)),
        "--max-order",
        str(int(max_order)),
        "--runs",
        str(int(runs)),
        "--warmup",
        str(int(warmup)),
        "--dtype",
        str(jnp.dtype(dtype)),
        "--seed",
        "0",
        "--autotune-cache",
        str(runtime_autotune_cache_path),
        "--config-json",
        json.dumps(payload),
    ]
    if emit_ready_marker:
        cmd.append("--emit-ready-marker")
    return cmd


def _parse_worker_json_output(lines):
    filtered = [line.strip() for line in lines if line and line.strip()]
    if not filtered:
        raise RuntimeError("worker produced no output")
    for line in reversed(filtered):
        if line == "__JACCPOT_WORKER_READY__":
            continue
        try:
            return json.loads(line)
        except json.JSONDecodeError:
            continue
    raise RuntimeError("worker produced no JSON output")


def _run_worker_case(mode, *, num_particles, leaf_size, max_order, runs, warmup, dtype, fmm_kwargs):
    _release_notebook_runtime_state()
    cmd = _build_worker_command(
        mode,
        num_particles=num_particles,
        leaf_size=leaf_size,
        max_order=max_order,
        runs=runs,
        warmup=warmup,
        dtype=dtype,
        fmm_kwargs=fmm_kwargs,
    )
    result = subprocess.run(cmd, check=False, capture_output=True, text=True)
    _release_notebook_runtime_state()
    if result.returncode != 0:
        details = (result.stderr or result.stdout or "").strip()
        raise RuntimeError(f"worker failed (exit={result.returncode}): {details}")
    return _parse_worker_json_output(result.stdout.splitlines())


def _run_worker_peak_case(mode, *, num_particles, leaf_size, max_order, dtype, fmm_kwargs, poll_interval_s):
    _release_notebook_runtime_state()
    cmd = _build_worker_command(
        mode,
        num_particles=num_particles,
        leaf_size=leaf_size,
        max_order=max_order,
        runs=1,
        warmup=0,
        dtype=dtype,
        fmm_kwargs=fmm_kwargs,
        emit_ready_marker=True,
    )
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    stdout_lines = []
    ready_seen = False
    while True:
        line = proc.stdout.readline()
        if line == "":
            if proc.poll() is not None:
                break
            continue
        stdout_lines.append(line.rstrip("\n"))
        if line.strip() == "__JACCPOT_WORKER_READY__":
            ready_seen = True
            break
    if not ready_seen:
        remaining = proc.stdout.read() if proc.stdout is not None else ""
        if remaining:
            stdout_lines.extend(remaining.splitlines())
        returncode = proc.wait()
        details = "\n".join(line for line in stdout_lines if line.strip())
        raise RuntimeError(f"worker failed before ready marker (exit={returncode}): {details.strip()}")

    baseline = _query_gpu_memory_mb_by_pid(proc.pid)
    t0 = time.perf_counter()
    trace_rows = []
    while proc.poll() is None:
        trace_rows.append(
            {
                "label": str(mode),
                "t_s": float(time.perf_counter()),
                "gpu_used_mb": _query_gpu_memory_mb_by_pid(proc.pid),
            }
        )
        time.sleep(float(poll_interval_s))
    t1 = time.perf_counter()
    after = _query_gpu_memory_mb_by_pid(proc.pid)
    remaining = proc.stdout.read() if proc.stdout is not None else ""
    if remaining:
        stdout_lines.extend(remaining.splitlines())
    returncode = proc.wait()
    _release_notebook_runtime_state()
    if returncode != 0:
        details = "\n".join(line for line in stdout_lines if line.strip())
        raise RuntimeError(f"worker failed (exit={returncode}): {details.strip()}")

    peak_samples = []
    if np.isfinite(baseline):
        peak_samples.append(float(baseline))
    peak_samples.extend(
        float(row["gpu_used_mb"]) for row in trace_rows if np.isfinite(float(row["gpu_used_mb"]))
    )
    peak = float(max(peak_samples)) if peak_samples else float("nan")
    summary = {
        "worker_pid": int(proc.pid),
        "wall_seconds": float(t1 - t0),
        "gpu_used_before_mb": float(baseline),
        "gpu_used_after_mb": float(after),
        "gpu_peak_used_mb": float(peak),
        "gpu_peak_delta_mb": (
            float(peak - baseline)
            if np.isfinite(peak) and np.isfinite(baseline)
            else float("nan")
        ),
        "num_samples": int(len(trace_rows)),
        "poll_interval_s": float(poll_interval_s),
    }
    row = _parse_worker_json_output(stdout_lines)
    trace_df = pd.DataFrame(trace_rows)
    return row, summary, trace_df


def _load_json_cache(path):
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text())
    except Exception:
        return {}


def _save_json_cache(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True))


def _apply_traversal_config(fmm_kwargs, traversal_cfg_dict):
    traversal_cfg = DualTreeTraversalConfig(
        max_pair_queue=int(traversal_cfg_dict["max_pair_queue"]),
        process_block=int(traversal_cfg_dict["process_block"]),
        max_interactions_per_node=int(traversal_cfg_dict["max_interactions_per_node"]),
        max_neighbors_per_leaf=int(traversal_cfg_dict["max_neighbors_per_leaf"]),
    )
    probe = FastMultipoleMethod(**fmm_kwargs)
    advanced = probe.advanced
    tuned_advanced = replace(
        advanced,
        runtime=replace(advanced.runtime, traversal_config=traversal_cfg),
    )
    _release_runtime_memory(probe)
    out = dict(fmm_kwargs)
    out["advanced"] = tuned_advanced
    return out


def _resolve_traversal_autotuned_kwargs(setup_name, fmm_kwargs):
    if not bool(runtime_enable_traversal_autotune):
        return fmm_kwargs, None

    cache = _load_json_cache(runtime_traversal_autotune_cache_path)
    entries = cache.get("entries", {}) if isinstance(cache, dict) else {}
    key = "|".join(
        [
            f"setup={setup_name}",
            f"leaf={int(runtime_leaf_size)}",
            f"order={int(runtime_max_order)}",
            f"dtype={str(jnp.dtype(runtime_working_dtype))}",
            f"theta={float(fmm_kwargs.get('theta', 0.6)):.4f}",
            f"idx={os.environ.get('JACCPOT_INDEX_PRECISION', 'int64')}",
        ]
    )
    cached = entries.get(key)
    if isinstance(cached, dict) and isinstance(cached.get("traversal_config"), dict):
        cfg = cached["traversal_config"]
        return _apply_traversal_config(fmm_kwargs, cfg), cfg

    probe_n = int(max(runtime_traversal_autotune_probe_n, max(autotune_warmup_counts)))
    selected_cfg = None
    for candidate in runtime_traversal_candidates:
        trial_kwargs = _apply_traversal_config(fmm_kwargs, candidate)
        try:
            row = _run_worker_case(
                "prepare",
                num_particles=probe_n,
                leaf_size=runtime_leaf_size,
                max_order=runtime_max_order,
                runs=1,
                warmup=0,
                dtype=runtime_working_dtype,
                fmm_kwargs=trial_kwargs,
            )
            prep_t = float(row.get("prepare_component_sum_seconds", float("nan")))
            if np.isfinite(prep_t):
                selected_cfg = {k: int(v) for k, v in candidate.items()}
                print(f"[traversal_autotune] setup={setup_name} accepted {selected_cfg} at N={probe_n}")
                break
        except Exception as exc:
            print(
                f"[traversal_autotune] setup={setup_name} candidate={candidate} failed: {type(exc).__name__}: {exc}"
            )

    if selected_cfg is None:
        print(
            f"[traversal_autotune] setup={setup_name} found no stable candidate; keeping preset traversal config"
        )
        return fmm_kwargs, None

    if not isinstance(cache, dict):
        cache = {}
    entries = cache.setdefault("entries", {})
    entries[key] = {
        "traversal_config": selected_cfg,
        "probe_n": int(probe_n),
    }
    _save_json_cache(runtime_traversal_autotune_cache_path, cache)
    return _apply_traversal_config(fmm_kwargs, selected_cfg), selected_cfg


def sweep_runtimes(
    particle_counts,
    *,
    leaf_size,
    max_order,
    runs,
    warmup,
    dtype,
    key,
    fmm_kwargs,
    strict=False,
):
    records = []
    use_subprocess = bool(globals().get("runtime_isolate_process_per_n", False))
    fmm = None if use_subprocess else FastMultipoleMethod(**fmm_kwargs)
    current_key = key

    for num_particles in particle_counts:
        try:
            if use_subprocess:
                records.append(
                    _run_worker_case(
                        "sweep",
                        num_particles=num_particles,
                        leaf_size=leaf_size,
                        max_order=max_order,
                        runs=runs,
                        warmup=warmup,
                        dtype=dtype,
                        fmm_kwargs=fmm_kwargs,
                    )
                )
                continue

            positions, masses, current_key = bench_utils.generate_random_distribution(
                num_particles,
                key=current_key,
                dtype=dtype,
            )

            benchmark_scope = str(globals().get("runtime_benchmark_scope", "steady_eval")).strip().lower()
            if benchmark_scope not in ("full", "steady_eval"):
                benchmark_scope = "steady_eval"

            prepare_once_timing = bench_utils.time_callable(
                fmm.prepare_state,
                positions,
                masses,
                leaf_size=leaf_size,
                max_order=max_order,
                warmup=warmup,
                runs=runs,
            )
            prepared_state = prepare_once_timing.result

            eval_kwargs = _evaluate_prepared_kwargs(fmm)
            eval_timing = bench_utils.time_callable(
                fmm.evaluate_prepared_state,
                prepared_state,
                warmup=warmup,
                runs=runs,
                **eval_kwargs,
            )

            if benchmark_scope == "full":
                full_timing = bench_utils.time_callable(
                    fmm.compute_accelerations,
                    positions,
                    masses,
                    leaf_size=leaf_size,
                    max_order=max_order,
                    reuse_prepared_state=False,
                    warmup=warmup,
                    runs=runs,
                )
                full_mean = full_timing.mean
                full_std = full_timing.std
            else:
                full_mean = eval_timing.mean
                full_std = eval_timing.std

            records.append(
                {
                    "num_particles": num_particles,
                    "mean_seconds": full_mean,
                    "std_seconds": full_std,
                    "prepare_mean_seconds": prepare_once_timing.mean,
                    "prepare_std_seconds": prepare_once_timing.std,
                    "evaluate_mean_seconds": eval_timing.mean,
                    "evaluate_std_seconds": eval_timing.std,
                    "benchmark_scope": benchmark_scope,
                    "error": "",
                }
            )
        except Exception as exc:
            msg = f"{type(exc).__name__}: {exc}"
            print(f"[sweep_runtimes] N={num_particles} failed: {msg}")
            records.append(
                {
                    "num_particles": num_particles,
                    "mean_seconds": float("nan"),
                    "std_seconds": float("nan"),
                    "prepare_mean_seconds": float("nan"),
                    "prepare_std_seconds": float("nan"),
                    "evaluate_mean_seconds": float("nan"),
                    "evaluate_std_seconds": float("nan"),
                    "error": msg,
                }
            )
            if strict:
                raise
        finally:
            if fmm is not None:
                _release_runtime_memory(fmm)

    return pd.DataFrame(records)


def profile_prepare_components(
    particle_counts,
    *,
    leaf_size,
    max_order,
    dtype,
    key,
    fmm_kwargs,
    runs,
    warmup,
    strict=False,
):
    records = []
    use_subprocess = bool(globals().get("runtime_isolate_process_per_n", False))
    fmm = None if use_subprocess else FastMultipoleMethod(**fmm_kwargs)
    current_key = key

    if not use_subprocess:
        tree_type = str(getattr(fmm._impl, "tree_type", "radix"))
        tree_mode = str(getattr(fmm._impl, "tree_build_mode", "lbvh"))
        ygg_build_mode = "fixed_depth" if tree_mode == "fixed_depth" else "adaptive"
        theta_val = float(getattr(fmm._impl, "theta", fmm_kwargs.get("theta", 0.6)))
        traversal_cfg = fmm.advanced.runtime.traversal_config
        mac_type = str(getattr(fmm, "mac_type", "dehnen"))
        dehnen_radius_scale = float(getattr(fmm._impl, "dehnen_radius_scale", 1.0))

    for num_particles in particle_counts:
        try:
            if use_subprocess:
                records.append(
                    _run_worker_case(
                        "prepare",
                        num_particles=num_particles,
                        leaf_size=leaf_size,
                        max_order=max_order,
                        runs=runs,
                        warmup=warmup,
                        dtype=dtype,
                        fmm_kwargs=fmm_kwargs,
                    )
                )
                continue

            positions, masses, current_key = bench_utils.generate_random_distribution(
                num_particles,
                key=current_key,
                dtype=dtype,
            )

            tree_timing = bench_utils.time_callable(
                Tree.from_particles,
                positions,
                masses,
                tree_type=tree_type,
                build_mode=ygg_build_mode,
                return_reordered=True,
                leaf_size=int(leaf_size),
                warmup=warmup,
                runs=runs,
            )

            tree = Tree.from_particles(
                positions,
                masses,
                tree_type=tree_type,
                build_mode=ygg_build_mode,
                return_reordered=True,
                leaf_size=int(leaf_size),
            )
            geometry = compute_tree_geometry(tree, tree.positions_sorted)

            interactions_timing = bench_utils.time_callable(
                build_interactions_and_neighbors,
                tree,
                geometry,
                theta=theta_val,
                traversal_config=traversal_cfg,
                mac_type=mac_type,
                dehnen_radius_scale=dehnen_radius_scale,
                warmup=warmup,
                runs=runs,
            )

            prepare_timing = bench_utils.time_callable(
                fmm.prepare_state,
                positions,
                masses,
                leaf_size=int(leaf_size),
                max_order=max_order,
                warmup=warmup,
                runs=runs,
            )

            residual = max(
                float(prepare_timing.mean)
                - float(tree_timing.mean)
                - float(interactions_timing.mean),
                0.0,
            )
            records.append(
                {
                    "num_particles": num_particles,
                    "tree_build_mean_seconds": float(tree_timing.mean),
                    "interactions_mean_seconds": float(interactions_timing.mean),
                    "upward_mean_seconds": residual,
                    "downward_mean_seconds": 0.0,
                    "prepare_component_sum_seconds": float(prepare_timing.mean),
                    "error": "",
                }
            )
        except Exception as exc:
            msg = f"{type(exc).__name__}: {exc}"
            print(f"[profile_prepare_components] N={num_particles} failed: {msg}")
            records.append(
                {
                    "num_particles": num_particles,
                    "tree_build_mean_seconds": float("nan"),
                    "upward_mean_seconds": float("nan"),
                    "interactions_mean_seconds": float("nan"),
                    "downward_mean_seconds": float("nan"),
                    "prepare_component_sum_seconds": float("nan"),
                    "error": msg,
                }
            )
            if strict:
                raise
        finally:
            if fmm is not None:
                _release_runtime_memory(fmm)

    return pd.DataFrame(records)

## Static Compiled Memory Analysis (100k)

This section compiles selected runtime phases and reports per-device static memory estimates from `compiled.memory_analysis()`.


In [ ]:
def _memory_stat_row(setup_name, phase, stats):
    temp = float(stats.temp_size_in_bytes)
    arg = float(stats.argument_size_in_bytes)
    out = float(stats.output_size_in_bytes)
    alias = float(stats.alias_size_in_bytes)
    total = temp + arg + out - alias
    print(f"[{setup_name}:{phase}] temp={temp/(1024**2):.2f} MB arg={arg/(1024**2):.2f} MB out={out/(1024**2):.2f} MB alias={alias/(1024**2):.2f} MB total={total/(1024**2):.2f} MB")
    return {
        "setup": str(setup_name),
        "phase": str(phase),
        "temp_mb": temp / (1024**2),
        "arg_mb": arg / (1024**2),
        "out_mb": out / (1024**2),
        "alias_mb": alias / (1024**2),
        "total_mb": total / (1024**2),
        "error": "",
    }


def _memory_block_ready(value):
    return jax.tree_util.tree_map(
        lambda x: x.block_until_ready() if hasattr(x, "block_until_ready") else x,
        value,
    )


def _compiled_memory_analysis_rows(setup_name, fmm_kwargs):
    rows = []
    key = jax.random.fold_in(runtime_key, int(runtime_memory_analysis_particles))
    positions, masses, _ = bench_utils.generate_random_distribution(
        int(runtime_memory_analysis_particles),
        key=key,
        dtype=runtime_working_dtype,
    )
    fmm = FastMultipoleMethod(**fmm_kwargs)
    try:
        if runtime_autotune_cache_path.exists():
            fmm.load_m2l_autotune_cache(str(runtime_autotune_cache_path), merge=True)

        # Outer jax.jit on prepare_state produces pathological compile-time
        # memory at large N. Measure the real prepare path eagerly instead.
        rows.append(
            {
                "setup": str(setup_name),
                "phase": "prepare",
                "temp_mb": np.nan,
                "arg_mb": np.nan,
                "out_mb": np.nan,
                "alias_mb": np.nan,
                "total_mb": np.nan,
                "error": "prepare_state memory_analysis skipped: outer jax.jit compile of prepare_state is disabled for large-N runtime measurements",
            }
        )

        state = fmm.prepare_state(
            positions,
            masses,
            leaf_size=int(runtime_leaf_size),
            max_order=int(runtime_max_order),
        )
        _ = _memory_block_ready(state)
        eval_kwargs = _evaluate_prepared_kwargs(fmm)

        # Evaluate compilation
        try:
            eval_jit = jax.jit(lambda st: fmm.evaluate_prepared_state(st, **eval_kwargs))
            eval_compiled = eval_jit.lower(state).compile()
            eval_stats = eval_compiled.memory_analysis()
            if eval_stats is not None:
                rows.append(_memory_stat_row(setup_name, "evaluate", eval_stats))
            del eval_compiled
        except Exception as exc:
            rows.append(
                {
                    "setup": str(setup_name),
                    "phase": "evaluate",
                    "temp_mb": np.nan,
                    "arg_mb": np.nan,
                    "out_mb": np.nan,
                    "alias_mb": np.nan,
                    "total_mb": np.nan,
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )

        if bool(runtime_memory_analysis_include_full):
            try:
                full_jit = jax.jit(
                    lambda p, m: fmm.compute_accelerations(
                        p,
                        m,
                        leaf_size=int(runtime_leaf_size),
                        max_order=int(runtime_max_order),
                        reuse_prepared_state=False,
                    )
                )
                full_compiled = full_jit.lower(positions, masses).compile()
                full_stats = full_compiled.memory_analysis()
                if full_stats is not None:
                    rows.append(_memory_stat_row(setup_name, "full", full_stats))
                del full_compiled
            except Exception as exc:
                rows.append(
                    {
                        "setup": str(setup_name),
                        "phase": "full",
                        "temp_mb": np.nan,
                        "arg_mb": np.nan,
                        "out_mb": np.nan,
                        "alias_mb": np.nan,
                        "total_mb": np.nan,
                        "error": f"{type(exc).__name__}: {exc}",
                    }
                )
    finally:
        _release_runtime_memory(fmm)
        jax.clear_caches()

    return rows


runtime_memory_df = pd.DataFrame()
if bool(runtime_memory_analysis_enabled):
    rows = []
    analysis_setups = runtime_setups
    for setup_name in runtime_memory_analysis_setups:
        if setup_name not in analysis_setups:
            continue
        rows.extend(_compiled_memory_analysis_rows(setup_name, analysis_setups[setup_name]))
    runtime_memory_df = pd.DataFrame(rows)

runtime_memory_df


## Prepared-State Runtime Memory Inventory

Runtime byte accounting for `prepare_state(...)` outputs (works even when static `prepare` compilation fails).


In [ ]:
def _array_nbytes_runtime(arr):
    try:
        shape = tuple(int(d) for d in getattr(arr, "shape", ()))
        dtype = np.dtype(getattr(arr, "dtype"))
    except Exception:
        return 0
    count = int(np.prod(np.asarray(shape, dtype=np.int64))) if len(shape) > 0 else 1
    return int(count * int(dtype.itemsize))


def _iter_array_paths(obj, prefix="root"):
    if hasattr(obj, "shape") and hasattr(obj, "dtype"):
        yield prefix, obj
        return
    if obj is None:
        return
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from _iter_array_paths(v, f"{prefix}.{k}")
        return
    if isinstance(obj, (list, tuple)):
        # NamedTuple support via _fields keeps paths readable.
        if hasattr(obj, "_fields"):
            for name in obj._fields:
                try:
                    v = getattr(obj, name)
                except Exception:
                    continue
                yield from _iter_array_paths(v, f"{prefix}.{name}")
            return
        for i, v in enumerate(obj):
            yield from _iter_array_paths(v, f"{prefix}[{i}]")
        return
    if hasattr(obj, "__dict__"):
        for k, v in vars(obj).items():
            yield from _iter_array_paths(v, f"{prefix}.{k}")
        return


def _subsystem_from_path(path):
    p = str(path)
    if ".upward" in p or ".multipoles" in p or ".mass_moments" in p:
        return "upward"
    if ".downward" in p or ".locals" in p:
        return "downward"
    if ".interactions" in p:
        return "interactions"
    if ".traversal_result" in p:
        return "traversal_result"
    if ".neighbor" in p or ".nearfield" in p:
        return "nearfield"
    if ".tree" in p or ".topology" in p or ".geometry" in p:
        return "tree_geometry"
    return "other"


def _prepared_state_memory_rows(setup_name, fmm_kwargs, n_particles):
    rows = []
    key = jax.random.fold_in(runtime_key, int(n_particles))
    positions, masses, _ = bench_utils.generate_random_distribution(
        int(n_particles),
        key=key,
        dtype=runtime_working_dtype,
    )
    fmm = FastMultipoleMethod(**fmm_kwargs)
    try:
        if runtime_autotune_cache_path.exists():
            fmm.load_m2l_autotune_cache(str(runtime_autotune_cache_path), merge=True)
        state = fmm.prepare_state(
            positions,
            masses,
            leaf_size=int(runtime_leaf_size),
            max_order=int(runtime_max_order),
        )
        state = jax.tree_util.tree_map(
            lambda x: x.block_until_ready() if hasattr(x, "block_until_ready") else x,
            state,
        )

        by_subsystem = {}
        by_path = []
        total_bytes = 0
        for path, arr in _iter_array_paths(state, prefix="prepared"):
            nb = _array_nbytes_runtime(arr)
            if nb <= 0:
                continue
            total_bytes += nb
            ss = _subsystem_from_path(path)
            by_subsystem[ss] = by_subsystem.get(ss, 0) + nb
            by_path.append((path, nb, ss))

        for ss, nb in sorted(by_subsystem.items(), key=lambda kv: kv[1], reverse=True):
            rows.append(
                {
                    "setup": str(setup_name),
                    "num_particles": int(n_particles),
                    "component": str(ss),
                    "mb": float(nb) / (1024**2),
                    "share_pct": (100.0 * float(nb) / float(total_bytes)) if total_bytes > 0 else np.nan,
                    "kind": "subsystem",
                    "error": "",
                }
            )

        top_paths = sorted(by_path, key=lambda x: x[1], reverse=True)[:20]
        for path, nb, ss in top_paths:
            rows.append(
                {
                    "setup": str(setup_name),
                    "num_particles": int(n_particles),
                    "component": str(path),
                    "mb": float(nb) / (1024**2),
                    "share_pct": (100.0 * float(nb) / float(total_bytes)) if total_bytes > 0 else np.nan,
                    "kind": f"top_path:{ss}",
                    "error": "",
                }
            )

        rows.append(
            {
                "setup": str(setup_name),
                "num_particles": int(n_particles),
                "component": "TOTAL_PREPARED_STATE",
                "mb": float(total_bytes) / (1024**2),
                "share_pct": 100.0,
                "kind": "total",
                "error": "",
            }
        )
    except Exception as exc:
        rows.append(
            {
                "setup": str(setup_name),
                "num_particles": int(n_particles),
                "component": "prepare_failed",
                "mb": np.nan,
                "share_pct": np.nan,
                "kind": "error",
                "error": f"{type(exc).__name__}: {exc}",
            }
        )
    finally:
        _release_runtime_memory(fmm)
        jax.clear_caches()
    return rows


prepared_memory_rows = []
for setup_name in runtime_prepared_memory_setups:
    if setup_name not in runtime_setups:
        continue
    for n_particles in runtime_prepared_memory_particles:
        prepared_memory_rows.extend(
            _prepared_state_memory_rows(
                setup_name,
                runtime_setups[setup_name],
                int(n_particles),
            )
        )

runtime_prepared_memory_df = pd.DataFrame(prepared_memory_rows)
runtime_prepared_memory_df


In [ ]:
# Full-table visibility helpers
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

runtime_prepared_memory_df = runtime_prepared_memory_df.sort_values(["setup", "num_particles", "kind", "mb"], ascending=[True, True, True, False])

# Subsystem summary (largest first)
runtime_prepared_memory_subsystems = runtime_prepared_memory_df[runtime_prepared_memory_df["kind"] == "subsystem"].sort_values(["num_particles", "mb"], ascending=[True, False])

# Top paths only (largest arrays)
runtime_prepared_memory_top_paths = runtime_prepared_memory_df[runtime_prepared_memory_df["kind"].str.startswith("top_path:")].sort_values(["num_particles", "mb"], ascending=[True, False])

# Persist full details for offline inspection
runtime_prepared_memory_csv_path = REPO_ROOT / "benchmarks" / "prepared_state_memory_inventory.csv"
runtime_prepared_memory_csv_path.parent.mkdir(parents=True, exist_ok=True)
runtime_prepared_memory_df.to_csv(runtime_prepared_memory_csv_path, index=False)
print(f"Wrote full prepared-state memory inventory to: {runtime_prepared_memory_csv_path}")

print("\n=== Subsystem Summary ===")
display(runtime_prepared_memory_subsystems)
print("\n=== Top Paths ===")
display(runtime_prepared_memory_top_paths)

## Peak GPU Memory Scaling

Measures warm `prepare_state` and warm `evaluate_prepared_state` runtime plus whole-process GPU peak memory as a function of particle count. This uses the same tuned runtime setups as the main scaling sweep.

In [ ]:
def _query_gpu_memory_mb_by_pid(pid):
    try:
        proc = subprocess.run(
            [
                "nvidia-smi",
                "--query-compute-apps=pid,used_memory",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            check=True,
        )
    except Exception:
        return float("nan")
    used = 0.0
    for line in proc.stdout.splitlines():
        parts = [part.strip() for part in line.split(",")]
        if len(parts) != 2:
            continue
        try:
            row_pid = int(parts[0])
            row_used = float(parts[1])
        except Exception:
            continue
        if row_pid == int(pid):
            used += row_used
    return float(used)


def _prepared_state_total_mb(state):
    total_bytes = 0
    for _path, arr in _iter_array_paths(state, prefix="prepared"):
        total_bytes += _array_nbytes_runtime(arr)
    return float(total_bytes) / (1024**2)


def _peak_gpu_memory_trace_runtime(fn, *args, label, poll_interval_s=0.02, **kwargs):
    import gc
    import os
    import threading

    rows = []
    stop_event = threading.Event()
    pid = os.getpid()

    def _poll():
        while not stop_event.is_set():
            rows.append(
                {
                    "label": str(label),
                    "t_s": float(time.perf_counter()),
                    "gpu_used_mb": _query_gpu_memory_mb_by_pid(pid),
                }
            )
            time.sleep(float(poll_interval_s))

    gc.collect()
    before = _query_gpu_memory_mb_by_pid(pid)
    thread = threading.Thread(target=_poll, daemon=True)
    thread.start()
    t0 = time.perf_counter()
    error = ""
    result = None
    try:
        result = fn(*args, **kwargs)
        result = _memory_block_ready(result)
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        raise
    finally:
        t1 = time.perf_counter()
        stop_event.set()
        thread.join(timeout=2.0)
    after = _query_gpu_memory_mb_by_pid(pid)
    trace_df = pd.DataFrame(rows)
    peak = float(trace_df["gpu_used_mb"].max()) if not trace_df.empty else float("nan")
    summary = {
        "label": str(label),
        "wall_seconds": float(t1 - t0),
        "gpu_used_before_mb": float(before),
        "gpu_used_after_mb": float(after),
        "gpu_peak_used_mb": peak,
        "gpu_peak_delta_mb": (
            float(peak - before)
            if np.isfinite(peak) and np.isfinite(before)
            else float("nan")
        ),
        "num_samples": int(len(trace_df)),
        "poll_interval_s": float(poll_interval_s),
        "error": str(error),
    }
    return result, pd.DataFrame([summary]), trace_df


def sweep_peak_memory_scaling(
    particle_counts,
    *,
    setup_names,
    leaf_size,
    max_order,
    dtype,
    key,
    poll_interval_s,
):
    del key
    rows = []
    trace_parts = []
    active_setups = runtime_setups_tuned if "runtime_setups_tuned" in globals() else runtime_setups
    for setup_name in setup_names:
        if setup_name not in active_setups:
            print(f"[peak_memory] setup '{setup_name}' not found; available={list(active_setups.keys())}")
            continue
        fmm_kwargs = active_setups[setup_name]
        for num_particles in particle_counts:
            row = {
                "setup": str(setup_name),
                "num_particles": int(num_particles),
                "prepared_state_mb": np.nan,
                "prepare_seconds": np.nan,
                "prepare_peak_gpu_delta_mb": np.nan,
                "prepare_peak_gpu_used_mb": np.nan,
                "evaluate_seconds": np.nan,
                "evaluate_peak_gpu_delta_mb": np.nan,
                "evaluate_peak_gpu_used_mb": np.nan,
                "error": "",
            }
            try:
                prepare_row, prepare_summary, prepare_trace_df = _run_worker_peak_case(
                    "peak_prepare",
                    num_particles=int(num_particles),
                    leaf_size=int(leaf_size),
                    max_order=int(max_order),
                    dtype=dtype,
                    fmm_kwargs=fmm_kwargs,
                    poll_interval_s=float(poll_interval_s),
                )
                eval_row, eval_summary, eval_trace_df = _run_worker_peak_case(
                    "peak_evaluate",
                    num_particles=int(num_particles),
                    leaf_size=int(leaf_size),
                    max_order=int(max_order),
                    dtype=dtype,
                    fmm_kwargs=fmm_kwargs,
                    poll_interval_s=float(poll_interval_s),
                )

                row["prepared_state_mb"] = float(
                    prepare_row.get(
                        "prepared_state_mb",
                        eval_row.get("prepared_state_mb", float("nan")),
                    )
                )
                row["prepare_seconds"] = float(prepare_row.get("prepare_seconds", prepare_summary["wall_seconds"]))
                row["prepare_peak_gpu_delta_mb"] = float(prepare_summary["gpu_peak_delta_mb"])
                row["prepare_peak_gpu_used_mb"] = float(prepare_summary["gpu_peak_used_mb"])
                row["evaluate_seconds"] = float(eval_row.get("evaluate_seconds", eval_summary["wall_seconds"]))
                row["evaluate_peak_gpu_delta_mb"] = float(eval_summary["gpu_peak_delta_mb"])
                row["evaluate_peak_gpu_used_mb"] = float(eval_summary["gpu_peak_used_mb"])

                for label, trace_df in (("prepare", prepare_trace_df), ("evaluate", eval_trace_df)):
                    if trace_df.empty:
                        continue
                    trace_copy = trace_df.copy()
                    trace_copy["label"] = str(label)
                    trace_copy["setup"] = str(setup_name)
                    trace_copy["num_particles"] = int(num_particles)
                    trace_parts.append(trace_copy)
                print(
                    f"[peak_memory] setup={setup_name} N={num_particles} prepare={row['prepare_seconds']:.3f}s peak_delta={row['prepare_peak_gpu_delta_mb']:.1f}MB eval={row['evaluate_seconds']:.3f}s"
                )
            except Exception as exc:
                row["error"] = f"{type(exc).__name__}: {exc}"
                print(f"[peak_memory] setup={setup_name} N={num_particles} failed: {row['error']}")
            rows.append(row)
    out_df = pd.DataFrame(rows)
    trace_df = pd.concat(trace_parts, ignore_index=True) if trace_parts else pd.DataFrame()
    return out_df, trace_df


In [ ]:
runtime_peak_memory_df = pd.DataFrame()
runtime_peak_memory_trace_df = pd.DataFrame()
if bool(runtime_peak_memory_enabled):
    runtime_peak_memory_df, runtime_peak_memory_trace_df = sweep_peak_memory_scaling(
        runtime_peak_memory_particle_counts,
        setup_names=runtime_peak_memory_setups,
        leaf_size=runtime_leaf_size,
        max_order=runtime_max_order,
        dtype=runtime_working_dtype,
        key=runtime_key,
        poll_interval_s=runtime_peak_memory_poll_interval_s,
    )
    runtime_peak_memory_csv_path.parent.mkdir(parents=True, exist_ok=True)
    runtime_peak_memory_df.to_csv(runtime_peak_memory_csv_path, index=False)
    runtime_peak_memory_trace_df.to_csv(runtime_peak_memory_trace_csv_path, index=False)
    print(f"Wrote runtime peak-memory summary to: {runtime_peak_memory_csv_path}")
    print(f"Wrote runtime peak-memory traces to: {runtime_peak_memory_trace_csv_path}")
runtime_peak_memory_df


In [ ]:
if runtime_peak_memory_df.empty:
    print("No runtime peak-memory data available.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for setup_name, sdf in runtime_peak_memory_df.groupby("setup"):
        sdf = sdf.sort_values("num_particles")
        ok = sdf[sdf["error"].astype(str) == ""]
        if ok.empty:
            continue
        axes[0].plot(ok["num_particles"], ok["prepare_seconds"], "o-", label=f"{setup_name}: prepare")
        axes[0].plot(ok["num_particles"], ok["evaluate_seconds"], "s--", label=f"{setup_name}: evaluate")
        axes[1].plot(ok["num_particles"], ok["prepare_peak_gpu_used_mb"], "o-", label=f"{setup_name}: prepare peak used")
        axes[1].plot(ok["num_particles"], ok["prepare_peak_gpu_delta_mb"], "s--", label=f"{setup_name}: prepare peak delta")
        axes[2].plot(ok["num_particles"], ok["prepared_state_mb"], "o-", label=f"{setup_name}: retained state")

    axes[0].set_xscale("log")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Number of particles")
    axes[0].set_ylabel("Seconds")
    axes[0].set_title("Warm prepare/evaluate runtime")
    axes[0].grid(True, which="both", linestyle=":", linewidth=0.8)
    axes[0].legend()

    axes[1].set_xscale("log")
    axes[1].set_yscale("log")
    axes[1].set_xlabel("Number of particles")
    axes[1].set_ylabel("GPU memory [MB]")
    axes[1].set_title("Warm prepare_state GPU peak")
    axes[1].grid(True, which="both", linestyle=":", linewidth=0.8)
    axes[1].legend()

    axes[2].set_xscale("log")
    axes[2].set_yscale("log")
    axes[2].set_xlabel("Number of particles")
    axes[2].set_ylabel("Prepared state retained MB")
    axes[2].set_title("Prepared-state retained size")
    axes[2].grid(True, which="both", linestyle=":", linewidth=0.8)
    axes[2].legend()

    plt.tight_layout()
    plt.show()


## Memory-Pressure Sweep (Knob Search)

Searches `m2l_chunk_size` and `nearfield_edge_chunk_size` combinations in isolated worker runs and reports stable non-OOM settings.


In [ ]:
def _with_memory_pressure_knobs(fmm_kwargs, *, m2l_chunk_size, nearfield_edge_chunk_size):
    probe = FastMultipoleMethod(**fmm_kwargs)
    adv = probe.advanced
    tuned_adv = replace(
        adv,
        farfield=replace(adv.farfield, m2l_chunk_size=int(m2l_chunk_size)),
        nearfield=replace(adv.nearfield, edge_chunk_size=int(nearfield_edge_chunk_size)),
        runtime=replace(adv.runtime, autotune_m2l_chunk=False),
    )
    _release_runtime_memory(probe)
    out = dict(fmm_kwargs)
    out["advanced"] = tuned_adv
    return out


def _is_oom_error(msg):
    text = str(msg).lower()
    return (
        "out of memory" in text
        or "resource_exhausted" in text
        or "cuda_error_out_of_memory" in text
        or "allocator" in text and "ran out of memory" in text
    )


memory_pressure_df = pd.DataFrame()
memory_pressure_best_df = pd.DataFrame()
if bool(runtime_memory_pressure_enabled):
    setup_name = str(runtime_memory_pressure_setup)
    base_setups = runtime_setups_tuned if "runtime_setups_tuned" in globals() else runtime_setups
    if setup_name not in base_setups:
        print(f"[memory_pressure] setup '{setup_name}' not found; available={list(base_setups.keys())}")
    else:
        base_kwargs = base_setups[setup_name]
        rows = []
        for n_particles in runtime_memory_pressure_particles:
            for m2l_chunk in runtime_memory_pressure_m2l_chunks:
                for nf_chunk in runtime_memory_pressure_nearfield_chunks:
                    trial_kwargs = _with_memory_pressure_knobs(
                        base_kwargs,
                        m2l_chunk_size=int(m2l_chunk),
                        nearfield_edge_chunk_size=int(nf_chunk),
                    )
                    row = {
                        "setup": setup_name,
                        "num_particles": int(n_particles),
                        "m2l_chunk_size": int(m2l_chunk),
                        "nearfield_edge_chunk_size": int(nf_chunk),
                        "mean_seconds": np.nan,
                        "prepare_mean_seconds": np.nan,
                        "evaluate_mean_seconds": np.nan,
                        "oom": False,
                        "error": "",
                    }
                    try:
                        out = _run_worker_case(
                            "sweep",
                            num_particles=int(n_particles),
                            leaf_size=int(runtime_leaf_size),
                            max_order=int(runtime_max_order),
                            runs=int(runtime_memory_pressure_runs),
                            warmup=int(runtime_memory_pressure_warmup),
                            dtype=runtime_working_dtype,
                            fmm_kwargs=trial_kwargs,
                        )
                        row["mean_seconds"] = float(out.get("mean_seconds", float("nan")))
                        row["prepare_mean_seconds"] = float(out.get("prepare_mean_seconds", float("nan")))
                        row["evaluate_mean_seconds"] = float(out.get("evaluate_mean_seconds", float("nan")))
                        err = str(out.get("error", ""))
                        row["error"] = err
                        row["oom"] = _is_oom_error(err)
                    except Exception as exc:
                        emsg = f"{type(exc).__name__}: {exc}"
                        row["error"] = emsg
                        row["oom"] = _is_oom_error(emsg)
                    rows.append(row)

        memory_pressure_df = pd.DataFrame(rows)
        if len(memory_pressure_df) > 0:
            ok = memory_pressure_df[
                (~memory_pressure_df["oom"]) & (memory_pressure_df["error"].fillna("") == "") & (np.isfinite(memory_pressure_df["mean_seconds"]))
            ].copy()
            if len(ok) > 0:
                memory_pressure_best_df = (
                    ok.sort_values(["num_particles", "mean_seconds", "m2l_chunk_size", "nearfield_edge_chunk_size"], ascending=[True, True, False, False])
                    .groupby("num_particles", as_index=False)
                    .first()
                )

print("=== Memory-Pressure Sweep Results ===")
display(memory_pressure_df)
print("=== Best Stable Knobs Per N ===")
display(memory_pressure_best_df)


## Runtime Scaling (Full vs Prepared Split)

In [ ]:
_release_notebook_runtime_state()

# 1) Resolve/persist traversal-capacity autotune per setup.
runtime_setups_tuned = {}
runtime_traversal_choices = {}
for setup_name, fmm_kwargs in runtime_setups.items():
    tuned_kwargs, chosen_cfg = _resolve_traversal_autotuned_kwargs(setup_name, fmm_kwargs)
    runtime_setups_tuned[setup_name] = tuned_kwargs
    runtime_traversal_choices[setup_name] = chosen_cfg
runtime_traversal_choices

# 2) Warm up M2L autotuning first so benchmark runs use persisted best chunk sizes.
for setup_name, fmm_kwargs in runtime_setups_tuned.items():
    for warm_n in autotune_warmup_counts:
        try:
            _ = _run_worker_case(
                "prepare",
                num_particles=warm_n,
                leaf_size=runtime_leaf_size,
                max_order=runtime_max_order,
                runs=1,
                warmup=0,
                dtype=runtime_working_dtype,
                fmm_kwargs=fmm_kwargs,
            )
        except Exception as exc:
            print(f"[autotune_warmup] setup={setup_name} N={warm_n} failed: {type(exc).__name__}: {exc}")

# 3) Runtime sweep for both setups.
runtime_results = {}
for setup_name, fmm_kwargs in runtime_setups_tuned.items():
    setup_df = sweep_runtimes(
        runtime_particle_counts,
        leaf_size=runtime_leaf_size,
        max_order=runtime_max_order,
        runs=runtime_runs,
        warmup=runtime_warmup,
        dtype=runtime_working_dtype,
        key=runtime_key,
        fmm_kwargs=fmm_kwargs,
    )
    setup_df["setup"] = setup_name
    runtime_results[setup_name] = setup_df.sort_values("num_particles")

runtime_compare_df = pd.concat(runtime_results.values(), ignore_index=True)
runtime_compare_df

In [ ]:
fig, (ax_rt, ax_sp) = plt.subplots(1, 2, figsize=(12, 4.5))

rt_has_positive = False
for setup_name, marker in [("fiducial", "o"), ("adaptive_error_mac", "s")]:
    df = runtime_results[setup_name]
    plot_df = df[np.isfinite(df["mean_seconds"]) & (df["mean_seconds"] > 0)].copy()
    if plot_df.empty:
        continue
    rt_has_positive = True
    yerr = plot_df["std_seconds"].where(np.isfinite(plot_df["std_seconds"]), 0.0)
    ax_rt.errorbar(
        plot_df["num_particles"],
        plot_df["mean_seconds"],
        yerr=yerr,
        fmt=f"{marker}-",
        capsize=4,
        label=f"{setup_name}: full compute",
    )

ax_rt.set_xscale("log")
if rt_has_positive:
    ax_rt.set_yscale("log")
ax_rt.set_xlabel("Number of particles")
ax_rt.set_ylabel("Runtime [s]")
ax_rt.set_title("Full runtime scaling by setup")
ax_rt.grid(True, which="both", linestyle=":", linewidth=0.8)
if rt_has_positive:
    ax_rt.legend()
else:
    ax_rt.text(0.5, 0.5, "No positive runtime results to plot", ha="center", va="center", transform=ax_rt.transAxes)

fid = runtime_results["fiducial"][["num_particles", "mean_seconds"]].rename(columns={"mean_seconds": "fiducial_s"})
ada = runtime_results["adaptive_error_mac"][["num_particles", "mean_seconds"]].rename(columns={"mean_seconds": "adaptive_s"})
speed = fid.merge(ada, on="num_particles", how="inner")
speed = speed[(speed["fiducial_s"] > 0) & (speed["adaptive_s"] > 0)].copy()
if not speed.empty:
    speed["speedup_x"] = speed["fiducial_s"] / speed["adaptive_s"]
else:
    speed["speedup_x"] = pd.Series(dtype=float)

if not speed.empty:
    ax_sp.plot(speed["num_particles"], speed["speedup_x"], "d-", color="tab:green")
ax_sp.axhline(1.0, color="gray", linestyle="--", linewidth=1)
ax_sp.set_xscale("log")
ax_sp.set_xlabel("Number of particles")
ax_sp.set_ylabel("Speedup vs fiducial [x]")
ax_sp.set_title("Adaptive+error-MAC speedup")
ax_sp.grid(True, which="both", linestyle=":", linewidth=0.8)
if speed.empty:
    ax_sp.text(0.5, 0.5, "No positive overlap between setups", ha="center", va="center", transform=ax_sp.transAxes)

plt.tight_layout()
plt.show()

speed

## Prepare-State Component Breakdown

In [ ]:
runtime_component_results = {}
for setup_name, fmm_kwargs in runtime_setups.items():
    comp_df = profile_prepare_components(
        runtime_particle_counts,
        leaf_size=runtime_leaf_size,
        max_order=runtime_max_order,
        dtype=runtime_working_dtype,
        key=runtime_key,
        fmm_kwargs=fmm_kwargs,
        runs=max(1, runtime_runs),
        warmup=runtime_warmup,
    )
    comp_df["setup"] = setup_name
    runtime_component_results[setup_name] = comp_df.sort_values("num_particles")

runtime_component_compare_df = pd.concat(runtime_component_results.values(), ignore_index=True)
runtime_component_compare_df

In [ ]:
fig, (ax_sum, ax_int) = plt.subplots(1, 2, figsize=(12, 4.5))
sum_has_positive = False
int_has_positive = False
for setup_name, marker in [("fiducial", "o"), ("adaptive_error_mac", "s")]:
    cdf = runtime_component_results[setup_name]
    sum_df = cdf[np.isfinite(cdf["prepare_component_sum_seconds"]) & (cdf["prepare_component_sum_seconds"] > 0)]
    if not sum_df.empty:
        sum_has_positive = True
        ax_sum.plot(
            sum_df["num_particles"],
            sum_df["prepare_component_sum_seconds"],
            marker=marker,
            label=f"{setup_name}: prepare total",
        )
    int_df = cdf[np.isfinite(cdf["interactions_mean_seconds"]) & (cdf["interactions_mean_seconds"] > 0)]
    if not int_df.empty:
        int_has_positive = True
        ax_int.plot(
            int_df["num_particles"],
            int_df["interactions_mean_seconds"],
            marker=marker,
            label=f"{setup_name}: interactions",
        )

for ax, title, ylabel, has_positive in [
    (ax_sum, "prepare_state total", "Runtime [s]", sum_has_positive),
    (ax_int, "interaction build", "Runtime [s]", int_has_positive),
]:
    ax.set_xscale("log")
    if has_positive:
        ax.set_yscale("log")
    ax.set_xlabel("Number of particles")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, which="both", linestyle=":", linewidth=0.8)
    if has_positive:
        ax.legend()
    else:
        ax.text(0.5, 0.5, "No positive values to plot", ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.show()

In [ ]:
runtime_index_precision = os.environ.get("JACCPOT_INDEX_PRECISION", "int64")

def _setup_traversal_cfg_value(setup_name, key):
    cfg = runtime_traversal_choices.get(str(setup_name)) if "runtime_traversal_choices" in globals() else None
    if not isinstance(cfg, dict):
        return np.nan
    val = cfg.get(str(key))
    return np.nan if val is None else int(val)

def _row_worker_traversal_cfg_value(worker_cfg, setup_name, key):
    if isinstance(worker_cfg, dict):
        val = worker_cfg.get(str(key))
        if val is not None:
            return int(val)
    return _setup_traversal_cfg_value(setup_name, key)

runtime_compare_df = runtime_compare_df.copy()
runtime_compare_df["index_precision"] = str(runtime_index_precision)
if "worker_traversal_config" not in runtime_compare_df.columns:
    runtime_compare_df["worker_traversal_config"] = None
if "worker_nearfield_edge_chunk_size" not in runtime_compare_df.columns:
    runtime_compare_df["worker_nearfield_edge_chunk_size"] = np.nan
if "benchmark_scope" not in runtime_compare_df.columns:
    runtime_compare_df["benchmark_scope"] = str(globals().get("runtime_benchmark_scope", "steady_eval"))

runtime_compare_df["traversal_tuned"] = runtime_compare_df.apply(
    lambda row: isinstance(row.get("worker_traversal_config"), dict)
    or isinstance((runtime_traversal_choices.get(str(row.get("setup"))) if "runtime_traversal_choices" in globals() else None), dict),
    axis=1,
)
for _key in [
    "max_pair_queue",
    "process_block",
    "max_interactions_per_node",
    "max_neighbors_per_leaf",
]:
    runtime_compare_df[f"traversal_{_key}"] = runtime_compare_df.apply(
        lambda row, k=_key: _row_worker_traversal_cfg_value(
            row.get("worker_traversal_config"),
            row.get("setup"),
            k,
        ),
        axis=1,
    )

runtime_compare_df["nearfield_edge_chunk_size"] = runtime_compare_df["worker_nearfield_edge_chunk_size"].map(
    lambda v: int(v) if pd.notna(v) else np.nan
)

runtime_config_summary_df = runtime_compare_df[
    [
        "setup",
        "num_particles",
        "index_precision",
        "traversal_tuned",
        "traversal_max_pair_queue",
        "traversal_process_block",
        "traversal_max_interactions_per_node",
        "traversal_max_neighbors_per_leaf",
        "nearfield_edge_chunk_size",
        "benchmark_scope",
        "worker_traversal_config",
        "worker_nearfield_edge_chunk_size",
        "mean_seconds",
        "prepare_mean_seconds",
        "evaluate_mean_seconds",
        "error",
    ]
].sort_values(["setup", "num_particles"])
runtime_config_summary_df

## Apples-To-Apples Timing Breakdown

This section separates cold prepare, hot prepare, hot evaluate, and full acceleration calls to compare end-to-end vs steady-state kernel throughput.

In [ ]:
def _block_ready(value):
    return jax.tree_util.tree_map(
        lambda x: x.block_until_ready() if hasattr(x, "block_until_ready") else x,
        value,
    )


apples_rows = []
apples_key = runtime_key
apples_setups = runtime_setups_tuned if "runtime_setups_tuned" in globals() else runtime_setups
for setup_name, fmm_kwargs in apples_setups.items():
    for n_particles in apples_particle_counts:
        positions, masses, apples_key = bench_utils.generate_random_distribution(
            int(n_particles),
            key=apples_key,
            dtype=runtime_working_dtype,
        )
        row = {
            "setup": setup_name,
            "num_particles": int(n_particles),
            "prepare_cold_seconds": np.nan,
            "prepare_hot_mean_seconds": np.nan,
            "evaluate_hot_mean_seconds": np.nan,
            "full_rebuild_mean_seconds": np.nan,
            "full_reuse_mean_seconds": np.nan,
            "eval_particles_per_sec": np.nan,
            "full_rebuild_particles_per_sec": np.nan,
            "error": "",
        }
        fmm = None
        try:
            fmm = FastMultipoleMethod(**fmm_kwargs)
            if runtime_autotune_cache_path.exists():
                fmm.load_m2l_autotune_cache(str(runtime_autotune_cache_path), merge=True)

            t0 = time.perf_counter()
            state_cold = fmm.prepare_state(
                positions,
                masses,
                leaf_size=runtime_leaf_size,
                max_order=runtime_max_order,
            )
            _ = _block_ready(state_cold)
            row["prepare_cold_seconds"] = float(time.perf_counter() - t0)

            prep_hot = bench_utils.time_callable(
                fmm.prepare_state,
                positions,
                masses,
                leaf_size=runtime_leaf_size,
                max_order=runtime_max_order,
                warmup=apples_warmup,
                runs=apples_runs,
            )
            row["prepare_hot_mean_seconds"] = float(prep_hot.mean)

            eval_hot = bench_utils.time_callable(
                fmm.evaluate_prepared_state,
                prep_hot.result,
                warmup=apples_warmup,
                runs=apples_runs,
                **_evaluate_prepared_kwargs(fmm),
            )
            row["evaluate_hot_mean_seconds"] = float(eval_hot.mean)

            full_rebuild = bench_utils.time_callable(
                fmm.compute_accelerations,
                positions,
                masses,
                leaf_size=runtime_leaf_size,
                max_order=runtime_max_order,
                reuse_prepared_state=False,
                warmup=apples_warmup,
                runs=apples_runs,
            )
            row["full_rebuild_mean_seconds"] = float(full_rebuild.mean)

            full_reuse = bench_utils.time_callable(
                fmm.compute_accelerations,
                positions,
                masses,
                leaf_size=runtime_leaf_size,
                max_order=runtime_max_order,
                reuse_prepared_state=True,
                warmup=apples_warmup,
                runs=apples_runs,
            )
            row["full_reuse_mean_seconds"] = float(full_reuse.mean)

            if np.isfinite(row["evaluate_hot_mean_seconds"]) and row["evaluate_hot_mean_seconds"] > 0:
                row["eval_particles_per_sec"] = float(
                    n_particles / row["evaluate_hot_mean_seconds"]
                )
            if np.isfinite(row["full_rebuild_mean_seconds"]) and row["full_rebuild_mean_seconds"] > 0:
                row["full_rebuild_particles_per_sec"] = float(
                    n_particles / row["full_rebuild_mean_seconds"]
                )
        except Exception as exc:
            row["error"] = f"{type(exc).__name__}: {exc}"
            print(f"[apples] setup={setup_name} N={n_particles} failed: {row['error']}")
        finally:
            if fmm is not None:
                _release_runtime_memory(fmm)
        apples_rows.append(row)

apples_df = pd.DataFrame(apples_rows).sort_values(["setup", "num_particles"])
apples_df

In [ ]:
apples_summary_df = apples_df[
    [
        "setup",
        "num_particles",
        "prepare_cold_seconds",
        "prepare_hot_mean_seconds",
        "evaluate_hot_mean_seconds",
        "full_rebuild_mean_seconds",
        "full_reuse_mean_seconds",
        "eval_particles_per_sec",
        "full_rebuild_particles_per_sec",
        "error",
    ]
]
apples_summary_df

## Accuracy Comparison (Vs Direct Sum)

Both setups are measured against a direct-sum reference. For large `N`, the reference is computed on a deterministic subset of targets to keep cost manageable.

In [ ]:
def _deterministic_sample_indices(n_items: int, cap: int | None) -> np.ndarray:
    if cap is None or cap <= 0 or n_items <= cap:
        return np.arange(n_items, dtype=np.int64)
    # Deterministic even-coverage sampling (no RNG variance in accuracy metrics).
    return np.linspace(0, n_items - 1, num=cap, dtype=np.int64)


def _direct_sum_accelerations_chunked(
    positions,
    masses,
    *,
    G: float,
    softening: float,
    target_indices,
    target_chunk: int,
    source_chunk: int,
):
    pos = np.asarray(positions)
    mass = np.asarray(masses)
    targets = np.asarray(target_indices, dtype=np.int64)
    out = np.zeros((targets.size, 3), dtype=pos.dtype)
    eps = np.finfo(pos.dtype).eps
    soft_sq = softening * softening

    for t0 in range(0, targets.size, target_chunk):
        t1 = min(t0 + target_chunk, targets.size)
        tidx = targets[t0:t1]
        p_t = pos[tidx]
        acc_t = np.zeros((tidx.size, 3), dtype=pos.dtype)

        for s0 in range(0, pos.shape[0], source_chunk):
            s1 = min(s0 + source_chunk, pos.shape[0])
            sidx = np.arange(s0, s1, dtype=np.int64)
            p_s = pos[s0:s1]
            m_s = mass[s0:s1]

            diff = p_t[:, None, :] - p_s[None, :, :]
            dist_sq = np.einsum("ijk,ijk->ij", diff, diff) + soft_sq
            dist = np.sqrt(dist_sq)
            inv_dist3 = 1.0 / (dist_sq * dist + eps)

            same = tidx[:, None] == sidx[None, :]
            inv_dist3[same] = 0.0

            weighted = m_s[None, :] * inv_dist3
            acc_t += -G * np.einsum("ij,ijk->ik", weighted, diff)

        out[t0:t1] = acc_t

    return out


accuracy_rows = []
acc_key = runtime_key
for n_particles in accuracy_particle_counts:
    positions, masses, acc_key = bench_utils.generate_random_distribution(
        int(n_particles),
        key=acc_key,
        dtype=runtime_working_dtype,
    )

    target_idx = _deterministic_sample_indices(int(n_particles), accuracy_direct_target_cap)
    acc_ref = _direct_sum_accelerations_chunked(
        positions,
        masses,
        G=1.0,
        softening=float(softening),
        target_indices=target_idx,
        target_chunk=int(accuracy_direct_target_chunk),
        source_chunk=int(accuracy_direct_source_chunk),
    )

    setup_fmms = {
        "fiducial": FastMultipoleMethod(**runtime_fmm_kwargs_fiducial),
        "adaptive_error_mac": FastMultipoleMethod(**runtime_fmm_kwargs_adaptive),
    }
    for setup_name, fmm in setup_fmms.items():
        if runtime_autotune_cache_path.exists():
            try:
                fmm.load_m2l_autotune_cache(str(runtime_autotune_cache_path), merge=True)
            except Exception as exc:
                print(f"[accuracy:{setup_name}] autotune cache load failed for N={n_particles}: {type(exc).__name__}: {exc}")

        acc_est = fmm.compute_accelerations(
            positions,
            masses,
            leaf_size=runtime_leaf_size,
            max_order=runtime_max_order,
            reuse_prepared_state=False,
        )
        acc_est_subset = np.asarray(acc_est)[target_idx]
        diff = acc_est_subset - acc_ref
        ref_norm = np.linalg.norm(acc_ref)
        rel_l2 = float(np.linalg.norm(diff) / max(ref_norm, 1e-12))
        denom = np.linalg.norm(acc_ref, axis=1)
        err = np.linalg.norm(diff, axis=1)
        rel_max = float(np.max(err / np.maximum(denom, 1e-12)))
        max_abs = float(np.max(np.abs(diff)))

        accuracy_rows.append(
            {
                "setup": setup_name,
                "num_particles": int(n_particles),
                "num_reference_targets": int(target_idx.size),
                "relative_l2": rel_l2,
                "relative_max": rel_max,
                "max_abs": max_abs,
            }
        )

        fmm.clear_runtime_caches(clear_jax_compilation=True)

accuracy_df = pd.DataFrame(accuracy_rows).sort_values(["setup", "num_particles"])
accuracy_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
for setup_name, sdf in accuracy_df.groupby("setup"):
    axes[0].plot(sdf["num_particles"], sdf["relative_l2"], "o-", label=setup_name)
    axes[1].plot(sdf["num_particles"], sdf["relative_max"], "s-", label=setup_name)

for ax, title, ylabel in [
    (axes[0], "Relative L2 vs direct sum", "Relative L2 error"),
    (axes[1], "Relative max vs direct sum", "Relative max error"),
]:
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Number of particles")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, which="both", linestyle=":", linewidth=0.8)
    ax.legend()

plt.tight_layout()
plt.show()

## Notes

- This notebook is GPU-focused and will fail early if no GPU is visible to JAX.
- `JACCPOT_INDEX_PRECISION` / `YGGDRAX_INDEX_PRECISION` is set at notebook start (`INDEX_PRECISION`) to switch index width (`int32`/`int64`). Restart kernel after changes.
- `runtime_autotune_cache_path` persists M2L chunk tuning across worker processes and future runs.
- `runtime_traversal_autotune_cache_path` persists traversal-capacity tuning (queue/block/neighbor/interaction caps) chosen from `runtime_traversal_candidates`.
- `runtime_runtime_autotune_cache_path` persists worker-selected runtime tuning (`worker_traversal_config`, `worker_nearfield_edge_chunk_size`) keyed by GPU/runtime signature.
- `runtime_benchmark_scope` defaults to `steady_eval` (hot prepared-state evaluation). Set to `full` for end-to-end rebuild timing.
- Runtime benchmarking compares two setups:
  - `fiducial` (`large_n_gpu` preset)
  - `adaptive_error_mac` (adaptive expansion order + error-controlled MAC)
- `runtime_worker_autotune_traversal` / `runtime_worker_autotune_nearfield_chunk` enable worker-side tuning (enabled by default in this notebook).
- Accuracy is measured against a chunked direct-sum reference. For large `N`, a deterministic target subset is used (`accuracy_direct_target_cap`).
- Run the notebook from the first cell so GPU/XLA environment flags are applied before JAX initializes.